# Estudio 06 · Fase 3 · 3c′: L + B1 + B2 en el modelo único

Enmienda `PHASE3_AMENDMENT` (commit `505b885`): decidida por Rosa **después de ver 3a1, 3a2 y 3b** y antes de
entrenar 3c. 3c′ = C3 + B1 + B2 · E3 · S4_s3 desde 2005, Platt por horizonte con 1e-6, contra L con la regla de 5
temporadas y `adopt_phase3`. B2 usa el núcleo de cada corte. Si empata o pierde, la fase 3 se cierra con L.


In [1]:
import json
import sys

import pandas as pd

from denguekay.config.paths import RESEARCH
from denguekay.evaluation import episodes as diag
from denguekay.model import tracking
from denguekay.model.device import xgboost_settings

STUDY = RESEARCH / "06_model_improvement"
sys.path.insert(0, str(STUDY / "code"))
import phase2_calibration as p2
import phase3_onsets as p3
import preregistration as pr

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
METRICS = STUDY / "docs" / "metricas" / "fase3c.json"
device = xgboost_settings()
inp = p3.phase3_inputs()
device


{'device': 'cuda', 'n_jobs': 12}

## 1 · Entrenamiento (5 semillas) y veredicto preregistrado


In [2]:
L = p3.reference_l(inp["serie"])
C = p3.run_3c_prime(inp, pr.SEEDS)
p3.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
pd.concat(C["predicciones"].values(), ignore_index=True).to_csv(p3.OUTPUT_DIR / "predicciones_3c_prima.csv.gz", index=False)
dec = p3.compare(C, L)
adopt = pr.adopt_phase3(dec["veredicto"])
print("Se adopta 3c′:", adopt)
dec["veredicto"]


Se adopta 3c′: False


,horizonte,veredicto,mejora_antelacion,empeora_antelacion,vetos,vetos_contra_referencia,a_favor_f1_050,a_favor_auprc,a_favor_brier,a_favor_razon_mae,en_contra_f1_050,en_contra_auprc,en_contra_brier,en_contra_razon_mae,a_favor_antelacion,en_contra_antelacion
0,2,empate,False,False,None,brier,0,1,3,1,2,2,2,1,1,0
1,3,empate,False,False,None,brier,0,2,3,1,2,1,1,2,2,1
2,4,empate,False,False,None,brier,0,1,3,1,2,2,1,2,2,0


In [3]:
dec["antelacion"].round(3)


,horizonte,episodios,diferencia,ic95_inferior,ic95_superior,diferencia_temporada_2010,diferencia_temporada_2015,diferencia_temporada_2016,diferencia_temporada_2022,diferencia_temporada_2023,episodios_temporada_2010,episodios_temporada_2015,episodios_temporada_2016,episodios_temporada_2022,episodios_temporada_2023
0,2,193,0.031,0.000,0.062,0.0,-0.029,0.000,0.026,0.105,30,34,34,38,57
1,3,193,0.016,-0.016,0.047,0.0,0.000,-0.059,0.053,0.053,30,34,34,38,57
2,4,193,0.031,0.000,0.062,0.0,0.000,0.088,0.000,0.053,30,34,34,38,57


In [4]:
dec["metricas"].round(3)


,horizonte,bloque,positivos_cand,f1_050_cand,f1_050_ref,recall_050_cand,recall_050_ref,precision_050_cand,precision_050_ref,auprc_cand,auprc_ref,brier_cand,brier_ref,razon_mae_cand,razon_mae_ref,razon_rmse_cand,razon_rmse_ref
3,2,temporada_2010,234.0,0.704,0.722,0.615,0.701,0.827,0.745,0.784,0.780,0.030,0.030,0.951,0.938,0.956,0.920
8,2,temporada_2015,557.0,0.747,0.772,0.632,0.695,0.902,0.865,0.887,0.885,0.054,0.051,0.984,0.945,1.092,1.053
9,2,temporada_2016,395.0,0.660,0.657,0.696,0.714,0.628,0.609,0.752,0.757,0.057,0.059,1.050,1.100,0.997,1.071
15,2,temporada_2022,431.0,0.708,0.712,0.684,0.696,0.734,0.728,0.780,0.781,0.053,0.054,0.969,0.962,1.035,1.010
16,2,temporada_2023,1039.0,0.855,0.857,0.833,0.828,0.881,0.889,0.947,0.947,0.062,0.062,0.845,0.847,0.867,0.940
17,2,temporada_2024,857.0,0.803,0.799,0.841,0.853,0.768,0.752,0.896,0.893,0.072,0.074,1.098,1.259,1.282,1.566
21,3,temporada_2010,234.0,0.585,0.631,0.457,0.551,0.803,0.732,0.711,0.706,0.036,0.036,0.907,0.897,0.915,0.891
26,3,temporada_2015,557.0,0.568,0.649,0.413,0.515,0.911,0.875,0.841,0.839,0.068,0.064,0.938,0.895,1.099,1.045
27,3,temporada_2016,395.0,0.626,0.627,0.623,0.658,0.628,0.599,0.699,0.709,0.064,0.065,1.055,1.131,1.017,1.142
33,3,temporada_2022,431.0,0.698,0.704,0.659,0.682,0.736,0.728,0.775,0.771,0.056,0.056,0.951,0.959,1.090,1.063


## 2 · Lecturas que no deciden


In [5]:
dec["lectura_sin_2010"]


,horizonte,veredicto,mejora_antelacion,empeora_antelacion,vetos,vetos_contra_referencia,a_favor_f1_050,a_favor_auprc,a_favor_brier,a_favor_razon_mae,en_contra_f1_050,en_contra_auprc,en_contra_brier,en_contra_razon_mae,a_favor_antelacion,en_contra_antelacion
0,2,empate,False,False,None,NaN,0,1,2,1,1,2,2,1,1,0
1,3,empate,False,False,None,brier,0,2,3,1,1,1,1,2,2,1
2,4,empate,False,False,None,brier,0,1,3,1,1,2,1,2,2,0


In [6]:
rd = {k: p2.readings(v, inp["serie"]) for k, v in (("L", L), ("3c_prima", C))}
pd.concat({k: v["recall_por_tipo"] for k, v in rd.items()}, names=["variante"]).round(3)


horizonte          bloque         tipo  semanas  recall  semanas_rango  recall_rango
variante                                                                                         
L        4           2  temporada_2010  brote_nuevo     51.0   0.118              0         0.059
         5           2  temporada_2010     en_curso    183.0   0.863              0         0.033
         14          2  temporada_2015  brote_nuevo     60.0   0.067              0         0.000
         15          2  temporada_2015     en_curso    497.0   0.771              0         0.014
         16          2  temporada_2016  brote_nuevo     53.0   0.132              0         0.000
         17          2  temporada_2016     en_curso    342.0   0.804              0         0.003
         18          2  temporada_2017  brote_nuevo     79.0   0.329              0         0.000
         19          2  temporada_2017     en_curso    460.0   0.815              0         0.004
         26          2  temporada_2022  brote_nuevo     49.0   0.122              0         0.041
         27          2  temporada_2022     en_curso    382.0   0.770              0         0.005
         28          2  temporada_2023  brote_nuevo     86.0   0.174              0         0.000
         29          2  temporada_2023     en_curso    953.0   0.887              0         0.001
         30          2  temporada_2024  brote_nuevo     34.0   0.353              0         0.000
         31          2  temporada_2024     en_curso    823.0   0.874              0         0.004
         36          3  temporada_2010  brote_nuevo     51.0   0.059              0         0.000
         37          3  temporada_2010     en_curso    183.0   0.689              0         0.022
         46          3  temporada_2015  brote_nuevo     60.0   0.033              0         0.000
         47          3  temporada_2015     en_curso    497.0   0.573              0         0.010
         48          3  temporada_2016  brote_nuevo     53.0   0.113              0         0.038
         49          3  temporada_2016     en_curso    342.0   0.743              0         0.003
         50          3  temporada_2017  brote_nuevo     79.0   0.228              0         0.000
         51          3  temporada_2017     en_curso    460.0   0.739              0         0.009
         58          3  temporada_2022  brote_nuevo     49.0   0.163              0         0.000
         59          3  temporada_2022     en_curso    382.0   0.749              0         0.010
         60          3  temporada_2023  brote_nuevo     86.0   0.186              0         0.000
         61          3  temporada_2023     en_curso    953.0   0.849              0         0.006
         62          3  temporada_2024  brote_nuevo     34.0   0.265              0         0.000
         63          3  temporada_2024     en_curso    823.0   0.888              0         0.005
         68          4  temporada_2010  brote_nuevo     51.0   0.020              0         0.000
         69          4  temporada_2010     en_curso    183.0   0.454              0         0.049
         78          4  temporada_2015  brote_nuevo     60.0   0.017              0         0.000
         79          4  temporada_2015     en_curso    497.0   0.364              0         0.012
         80          4  temporada_2016  brote_nuevo     53.0   0.057              0         0.000
         81          4  temporada_2016     en_curso    342.0   0.652              0         0.006
         82          4  temporada_2017  brote_nuevo     79.0   0.203              0         0.013
         83          4  temporada_2017     en_curso    460.0   0.654              0         0.007
         90          4  temporada_2022  brote_nuevo     49.0   0.122              0         0.020
         91          4  temporada_2022     en_curso    382.0   0.660              0         0.024
         92          4  temporada_2023  brote_nuevo     86.0   0.174              0         0

In [7]:
pd.concat({k: v["hu0007_4"] for k, v in rd.items()}, names=["variante"])


horizonte          bloque  precision_050  recall_050    f1_050  razon_mae  razon_rmse  cumple_precision_050  cumple_recall_050  cumple_f1_050  cumple_razon_mae  cumple_razon_rmse  cumple_clasificacion  \
variante                                                                                                                                                                                                                
L        0           2  temporada_2010       0.745455    0.700855  0.722467   0.938085    0.919739                  True              False           True             False              False                 False   
         1           2  temporada_2015       0.865169    0.694794  0.772455   0.945156    1.053429                  True              False           True             False              False                 False   
         2           2  temporada_2016       0.609071    0.713924  0.657343   1.100044    1.071417                  True              False          False             False              False                 False   
         3           2  temporada_2017       0.663921    0.743970  0.703316   0.800489    0.875387                  True              False           True              True              False                 False   
         4           2  temporada_2022       0.728155    0.696056  0.711905   0.961816    1.010115                  True              False           True             False              False                 False   
         5           2  temporada_2023       0.888545    0.827719  0.857428   0.846863    0.940025                  True               True           True              True              False                  True   
         6           2  temporada_2024       0.752058    0.852975  0.799344   1.259388    1.565843                  True               True           True             False              False                  True   
         7           3  temporada_2010       0.731844    0.551282  0.630807   0.896763    0.890730                  True              False          False             False              False                 False   
         8           3  temporada_2015       0.875000    0.515260  0.648588   0.894666    1.044787                  True              False          False             False              False                 False   
         9           3  temporada_2016       0.599078    0.658228  0.627262   1.130717    1.142470                 False              False          False             False              False                 False   
         10          3  temporada_2017       0.652095    0.664193  0.658065   0.754253    0.840480                  True              False          False              True               True                 False   
         11          3  temporada_2022       0.727723    0.682135  0.704327   0.959157    1.063237                  True              False           True             False              False                 False   
         12          3  temporada_2023       0.880342    0.794033  0.835020   0.790613    0.887695                  True              False           True              True              False                 False   
         13          3  temporada_2024       0.725221    0.863477  0.787814   1.382340    1.777912                  True               True           True             False              False                  True   
         14          4  temporada_2010       0.714286    0.358974  0.475921   0.875766    0.858832                  True              False          False             False              False                 False   
         15          4  temporada_2015       0.900990    0.326750  0.479578   0.846978    0.993067                  True              False          False              True              False                 False   
         16          4  temporada_2016       0.608108    0.572152  0.590377   1.150254    1.175559         

In [8]:
pd.concat({k: v["antelacion_semilla_17"] for k, v in rd.items()}, names=["variante"]).round(3)


horizonte          bloque               alerta  episodios  avisados_2_o_mas  ic95_inferior  ic95_superior
variante                                                                                                               
L        2            2  temporada_2010           alerta_050         30             0.033          0.000          0.100
         7            2  temporada_2015           alerta_050         34             0.059          0.000          0.147
         8            2  temporada_2016           alerta_050         34             0.206          0.088          0.353
         9            2  temporada_2017           alerta_050         56             0.446          0.321          0.571
         13           2  temporada_2022           alerta_050         38             0.132          0.026          0.237
         14           2  temporada_2023           alerta_050         57             0.193          0.105          0.298
         15           2  temporada_2024           alerta_050         23             0.435          0.217          0.652
         19           3  temporada_2010           alerta_050         30             0.067          0.000          0.167
         24           3  temporada_2015           alerta_050         34             0.029          0.000          0.088
         25           3  temporada_2016           alerta_050         34             0.265          0.118          0.412
         26           3  temporada_2017           alerta_050         56             0.446          0.321          0.571
         30           3  temporada_2022           alerta_050         38             0.184          0.079          0.316
         31           3  temporada_2023           alerta_050         57             0.316          0.211          0.439
         32           3  temporada_2024           alerta_050         23             0.565          0.348          0.783
         36           4  temporada_2010           alerta_050         30             0.033          0.000          0.100
         41           4  temporada_2015           alerta_050         34             0.029          0.000          0.088
         42           4  temporada_2016           alerta_050         34             0.176          0.059          0.324
         43           4  temporada_2017           alerta_050         56             0.411          0.286          0.536
         47           4  temporada_2022           alerta_050         38             0.158          0.053          0.289
         48           4  temporada_2023           alerta_050         57             0.368          0.246          0.491
         49           4  temporada_2024           alerta_050         23             0.522          0.304          0.739
         53           2  temporada_2010  alerta_persistencia         30             0.033          0.000          0.100
         58           2  temporada_2015  alerta_persistencia         34             0.059          0.000          0.147
         59           2  temporada_2016  alerta_persistencia         34             0.118          0.029          0.235
         60           2  temporada_2017  alerta_persistencia         56             0.161          0.071          0.250
         64           2  temporada_2022  alerta_persistencia         38             0.105          0.026          0.211
         65           2  temporada_2023  alerta_persistencia         57             0.123          0.053          0.211
         66           2  temporada_2024  alerta_persistencia         23             0.130          0.000          0.261
         70           3  temporada_2010  alerta_persistencia         30             0.067          0.000          0.167
         75           3  temporada_2015  alerta_persistencia         34             0.059          0.000          0.147
         76           3  temporada_2016  alerta_persistencia         34             0.176          0.059          0.324
         77           3  temporada_2017  alerta_persis

## 3 · Salidas: métricas y MLflow


In [9]:
to_json = lambda d: {k: diag.to_json(v) for k, v in d.items() if isinstance(v, pd.DataFrame)}
content = {"dispositivo": device, "veredicto": to_json(dec), "adopta_3c_prima": adopt,
           "lecturas": {k: to_json(v) for k, v in rd.items()}}
content = json.loads(json.dumps(content, default=lambda o: o.item() if hasattr(o, "item") else str(o)))
for section, value in content.items():
    diag.save_metrics_section(METRICS, section, value)
tracking.log_study(
    "mejora_modelo_fase3_inicios", pd.DataFrame(columns=["numero", "estado"]),
    parent={"params": {"fase": "3c_prima", "enmienda": "505b885", "adopta": str(adopt),
                       "veredicto": "; ".join(f"h{r.horizonte}: {r.veredicto}" for r in dec["veredicto"].itertuples()),
                       **{f"xgb.{k}": str(x) for k, x in device.items()}},
            "metricas": {}},
    artifacts=[METRICS], tables={"veredicto_3c_prima": dec["veredicto"]},
    description="Fase 3 del estudio 06, enmienda 3c′: L + B1 + B2 en el modelo único.")


C:\Users\Rosa\RufoEsEternoElRegreso\Denguekay\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MLflow: estudio 588ac341bfed43dfaa4ba2190898426b en 'denguekay/mejora_modelo_fase3_inicios' (sqlite:///C:/Users/Rosa/RufoEsEternoElRegreso/Denguekay/mlflow.db)


'588ac341bfed43dfaa4ba2190898426b'